In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model.foragers import ForagerCollection


forager_collection: ForagerCollection = ForagerCollection()
df = forager_collection.get_all_foragers()

df


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model.foragers import ForagerCollection


forager_collection: ForagerCollection = ForagerCollection()
df = forager_collection.get_all_foragers()

ground_truth_forager_alias= "QLearning_L1F1_CK1_softmax" # bari
fitting_forager_alias= ["QLearning_L1F1_CK1_softmax","ForagingCompareThreshold_L2_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF"]



In [ ]:
"""
Parameter analysis for fitted dynamic-foraging models.

This script loads fitted model parameters from session-level JSON files,
summarizes their distributions, and visualizes them across sessions and
experimental protocols.

Expected directory structure
----------------------------
ROOT/
    <session_id>/
        <model_name>/
            <model_name>.json

Each JSON file should contain fitted parameter values and metadata such as:

{
    "fit_names": [...],          # names of fitted parameters
    "x": [...],                  # fitted values corresponding to fit_names
    "protocol": "...",           # experimental protocol name
    "auto_train_stage": "...",   # training stage of the session
    "n_trials": int,             # number of trials in the session
    "LPT": float,                # likelihood-per-trial
    "AIC": float,                # Akaike Information Criterion
    "BIC": float,                # Bayesian Information Criterion
    "prediction_accuracy": float
}

Main functionality
------------------
1. Load fitted parameters for a given model across all sessions.
2. Extract parameter columns automatically (excluding metadata fields).
3. Compute summary statistics for each parameter:
       - mean
       - standard deviation
       - median
       - quantiles (0.1%, 1%, 5%, 25%, 75%, 95%, 99%, 99.9%)
       - min / max
4. Compute the same summaries separately for each protocol.
5. Visualize parameter distributions:
       - overall distributions across all sessions
       - distributions separated by protocol

Outputs
-------
Printed tables:
    - preview of loaded sessions
    - protocol counts
    - parameter summary across sessions
    - parameter summary by protocol

Figures:
    - histogram distributions for each parameter
    - parameter histograms separated by protocol

Usage
-----
Set the ROOT directory and MODEL_NAME in the configuration section.
Optionally specify SELECTED_PROTOCOLS to visualize only a subset of protocols.

Example:
    MODEL_NAME = "Bari2019"
    SELECTED_PROTOCOLS = ["Coupled Baiting", "Uncoupled Baiting"]
"""

from pathlib import Path
import json
import math
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# User configuration
# --------------------------------------------------

ROOT = Path("/root/capsule/scratch/model_comparison")
MODEL_NAME = "Bari2019"

# Set to None to use all protocols found in the data
# Example:
# SELECTED_PROTOCOLS = ["Coupled Baiting", "Uncoupled Baiting"]
SELECTED_PROTOCOLS = None

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def _is_valid_number(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)

def load_model_params(root: Path, model_name: str) -> pd.DataFrame:
    """
    Load fitted parameters from:
        root / <session_id> / <model_name> / <model_name>.json

    Expected JSON structure:
        {
          "fit_names": [...],
          "x": [...]
        }
    """

    rows = []
    skipped = []

    for session_dir in sorted(root.iterdir()):
        if not session_dir.is_dir():
            continue

        json_path = session_dir / model_name / f"{model_name}.json"
        if not json_path.exists():
            continue

        try:
            with open(json_path, "r") as f:
                data = json.load(f)
        except Exception as e:
            skipped.append((str(json_path), f"json load failed: {e}"))
            continue

        fit_names = data.get("fit_names", None)
        x = data.get("x", None)

        if not isinstance(fit_names, list) or not isinstance(x, list):
            skipped.append((str(json_path), "missing fit_names or x"))
            continue

        if len(fit_names) != len(x):
            skipped.append(
                (str(json_path), f"length mismatch: len(fit_names)={len(fit_names)}, len(x)={len(x)}")
            )
            continue

        row = {
            "session": session_dir.name,
            "protocol": data.get("protocol"),
            "auto_train_stage": data.get("auto_train_stage"),
            "n_trials": data.get("n_trials"),
            "LPT": data.get("LPT"),
            "AIC": data.get("AIC"),
            "BIC": data.get("BIC"),
            "prediction_accuracy": data.get("prediction_accuracy"),
        }

        for name, value in zip(fit_names, x):
            row[name] = value if _is_valid_number(value) else float("nan")

        rows.append(row)

    df = pd.DataFrame(rows)

    print(f"Loaded {len(df)} sessions for model: {model_name}")
    print(f"Skipped {len(skipped)} files")

    if skipped:
        print("\nFirst 10 skipped files:")
        for path, reason in skipped[:10]:
            print(f"  {path} --> {reason}")

    return df

def get_parameter_columns(df: pd.DataFrame):
    """
    Return fitted-parameter columns by excluding metadata columns.
    """
    meta_cols = {
        "session", "protocol", "auto_train_stage", "n_trials",
        "LPT", "AIC", "BIC", "prediction_accuracy"
    }
    return [c for c in df.columns if c not in meta_cols]

def summarize_parameters(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute summary statistics for fitted parameters across all sessions.
    """

    param_cols = get_parameter_columns(df)
    summary = []

    for col in param_cols:
        vals = pd.to_numeric(df[col], errors="coerce").dropna()

        if len(vals) == 0:
            continue

        summary.append({
            "parameter": col,
            "n": len(vals),
            "mean": vals.mean(),
            "std": vals.std(),
            "median": vals.median(),
            "p001": vals.quantile(0.001),   # 0.1 percentile
            "p01": vals.quantile(0.01),     # 1 percentile
            "p05": vals.quantile(0.05),
            "p25": vals.quantile(0.25),
            "p75": vals.quantile(0.75),
            "p95": vals.quantile(0.95),
            "p99": vals.quantile(0.99),     # 99 percentile
            "p999": vals.quantile(0.999),   # 99.9 percentile
            "min": vals.min(),
            "max": vals.max(),
        })

    return pd.DataFrame(summary)

def summarize_parameters_by_protocol(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute summary statistics for fitted parameters within each protocol.
    """

    param_cols = get_parameter_columns(df)
    summary = []

    for protocol, subdf in df.groupby("protocol", dropna=False):
        for col in param_cols:
            vals = pd.to_numeric(subdf[col], errors="coerce").dropna()

            if len(vals) == 0:
                continue

            summary.append({
                "protocol": protocol,
                "parameter": col,
                "n": len(vals),
                "mean": vals.mean(),
                "std": vals.std(),
                "median": vals.median(),
                "p001": vals.quantile(0.001),   # 0.1 percentile
                "p01": vals.quantile(0.01),     # 1 percentile
                "p05": vals.quantile(0.05),
                "p25": vals.quantile(0.25),
                "p75": vals.quantile(0.75),
                "p95": vals.quantile(0.95),
                "p99": vals.quantile(0.99),     # 99 percentile
                "p999": vals.quantile(0.999),   # 99.9 percentile
                "min": vals.min(),
                "max": vals.max(),
            })

    return pd.DataFrame(summary)

def plot_parameter_distributions(df: pd.DataFrame, model_name: str, bins: int = 30):
    """
    Plot overall parameter distributions across all sessions.
    """
    param_cols = get_parameter_columns(df)

    if len(param_cols) == 0:
        print("No parameter columns found.")
        return

    fig, axes = plt.subplots(
        nrows=len(param_cols),
        ncols=1,
        figsize=(7, 3.2 * len(param_cols)),
        squeeze=False,
    )

    for ax, col in zip(axes[:, 0], param_cols):
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        ax.hist(vals, bins=bins)
        ax.axvline(vals.mean(), linestyle="--")
        ax.set_title(f"{model_name}: {col} (all protocols, n={len(vals)})")
        ax.set_xlabel(col)
        ax.set_ylabel("count")

    plt.tight_layout()
    plt.show()

def plot_parameter_distributions_by_protocol(
    df: pd.DataFrame,
    model_name: str,
    bins: int = 30,
    selected_protocols=None,
):
    """
    Plot separate parameter distributions for each protocol.

    Layout:
        rows = parameters
        cols = protocols
    """

    if df.empty:
        print("Input DataFrame is empty.")
        return

    plot_df = df.copy()

    if selected_protocols is not None:
        plot_df = plot_df[plot_df["protocol"].isin(selected_protocols)].copy()

    protocols = list(pd.Series(plot_df["protocol"]).dropna().unique())

    if len(protocols) == 0:
        print("No protocols found after filtering.")
        return

    param_cols = get_parameter_columns(plot_df)

    if len(param_cols) == 0:
        print("No parameter columns found.")
        return

    fig, axes = plt.subplots(
        nrows=len(param_cols),
        ncols=len(protocols),
        figsize=(4.8 * len(protocols), 3.2 * len(param_cols)),
        squeeze=False,
        sharex=False,
        sharey=False,
    )

    for i, param in enumerate(param_cols):
        for j, protocol in enumerate(protocols):
            ax = axes[i, j]
            vals = pd.to_numeric(
                plot_df.loc[plot_df["protocol"] == protocol, param],
                errors="coerce"
            ).dropna()

            if len(vals) > 0:
                ax.hist(vals, bins=bins)
                ax.axvline(vals.mean(), linestyle="--")
                ax.set_title(f"{protocol}\n{param} (n={len(vals)})")
            else:
                ax.set_title(f"{protocol}\n{param} (n=0)")

            ax.set_xlabel(param)
            ax.set_ylabel("count")

    fig.suptitle(f"{model_name}: parameter distributions by protocol", y=1.02)
    plt.tight_layout()
    plt.show()
# --------------------------------------------------
# Run
# --------------------------------------------------

df_params = load_model_params(ROOT, MODEL_NAME)

print("\nPreview:")
display(df_params.head())

print("\nProtocol counts:")
display(df_params["protocol"].value_counts(dropna=False).rename("count").to_frame())

print("\nOverall parameter summary:")
summary_df = summarize_parameters(df_params)
display(summary_df)

print("\nParameter summary by protocol:")
summary_by_protocol_df = summarize_parameters_by_protocol(df_params)
display(summary_by_protocol_df)

# Overall distributions
plot_parameter_distributions(df_params, MODEL_NAME, bins=30)

# Separate distributions by protocol
plot_parameter_distributions_by_protocol(
    df_params,
    MODEL_NAME,
    bins=30,
    selected_protocols=SELECTED_PROTOCOLS,
)

In [1]:
from __future__ import annotations

import copy
import itertools
import json
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cloudpickle
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model.foragers import ForagerCollection





# =============================================================================
# Helpers
# =============================================================================

def _make_task(
    *,
    use_uncoupled: bool,
    num_trials: int,
    seed: int,
    reward_baiting: bool,
):
    """Build the synthetic dynamic-foraging task."""
    if use_uncoupled:
        return UncoupledBlockTask(
            reward_baiting=reward_baiting,
            num_trials=num_trials,
            seed=seed,
        )
    return CoupledBlockTask(
        reward_baiting=reward_baiting,
        num_trials=num_trials,
        seed=seed,
    )


def _safe_model_dump_params(agent) -> Dict[str, Any]:
    """Return params as a plain dict when possible."""
    if hasattr(agent, "params") and hasattr(agent.params, "model_dump"):
        return dict(agent.params.model_dump())
    if hasattr(agent, "params") and isinstance(agent.params, dict):
        return dict(agent.params)
    return {}


def _to_jsonable(obj: Any) -> Any:
    """Convert nested objects into JSON-serializable objects."""
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def _sanitize_name(text: str) -> str:
    """Sanitize a string for filesystem paths."""
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def _clone_forager_from_df(df: pd.DataFrame, alias: str, seed: int = 42):
    """
    Fetch a fresh forager object from df['agent_alias'] / df['forager'].
    """
    rows = df.loc[df["agent_alias"] == alias]
    if len(rows) == 0:
        raise ValueError(f"Alias not found in df['agent_alias']: {alias}")
    if len(rows) > 1:
        raise ValueError(f"Alias is not unique in df['agent_alias']: {alias}")

    obj = rows.iloc[0]["forager"]

    if hasattr(obj, "fit") and hasattr(obj, "perform"):
        agent = copy.deepcopy(obj)
        if hasattr(agent, "seed"):
            try:
                agent.seed = seed
            except Exception:
                pass
        return agent

    if callable(obj):
        try:
            return obj(seed=seed)
        except TypeError:
            try:
                return obj()
            except Exception as e:
                raise RuntimeError(
                    f"Could not instantiate forager from callable for alias={alias}"
                ) from e

    raise TypeError(
        f"Unsupported object type in df['forager'] for alias={alias}: {type(obj)}"
    )


def _set_params_if_requested(agent, params_override: Optional[Dict[str, Any]]) -> None:
    """Set generator parameters if an override dict is provided."""
    if not params_override:
        return
    if not hasattr(agent, "set_params"):
        raise AttributeError("Agent does not have set_params(...)")
    agent.set_params(**params_override)


def _run_agent_on_observed_history(agent, choice_history, reward_history):
    """
    Replay the observed history through the fitted agent so latent variables
    needed by plotting are populated.
    """
    if hasattr(agent, "predict"):
        try:
            agent.predict(choice_history, reward_history)
            return
        except Exception:
            pass

    if hasattr(agent, "perform_closed_loop"):
        try:
            agent.perform_closed_loop(choice_history, reward_history)
            return
        except Exception:
            pass

    if hasattr(agent, "choice_history"):
        try:
            agent.choice_history = np.asarray(choice_history)
        except Exception:
            pass

    if hasattr(agent, "reward_history"):
        try:
            agent.reward_history = np.asarray(reward_history)
        except Exception:
            pass


def _fit_agent_on_data(
    *,
    agent,
    agent_alias: str,
    choice_history,
    reward_history,
    seed: int,
    workers: int,
) -> Dict[str, Any]:
    """Fit one agent on the same dataset and return a summary dict."""
    agent.fit(
        choice_history,
        reward_history,
        clamp_params={},
        DE_kwargs=dict(
            workers=workers,
            disp=False,
            seed=np.random.default_rng(seed),
        ),
        k_fold_cross_validation=None,
    )

    fr = agent.fitting_result
    fit_names = list(fr.fit_settings["fit_names"])
    fitted_params = dict(zip(fit_names, fr.x))

    _run_agent_on_observed_history(agent, choice_history, reward_history)

    out = dict(
        agent_alias=agent_alias,
        fit_names=fit_names,
        fitted_params=fitted_params,
        LPT=float(fr.LPT),
        prediction_accuracy=float(fr.prediction_accuracy),
        agent=agent,
    )

    # Add extra fit summary fields if available
    for attr in ["AIC", "BIC"]:
        if hasattr(fr, attr):
            try:
                out[attr] = float(getattr(fr, attr))
            except Exception:
                pass

    if hasattr(fr, "fit_bounds"):
        out["fit_bounds"] = _to_jsonable(fr.fit_bounds)
    elif hasattr(fr, "fit_settings") and isinstance(fr.fit_settings, dict):
        if "fit_bounds" in fr.fit_settings:
            out["fit_bounds"] = _to_jsonable(fr.fit_settings["fit_bounds"])

    return out


def _make_param_comparison_rows(
    *,
    ground_truth_alias: str,
    ground_truth_params: Dict[str, Any],
    results: List[Dict[str, Any]],
    combo_id: int,
    repeat_idx: int,
    task_seed: int,
    model_seed: int,
) -> List[Dict[str, Any]]:
    """Build rows comparing fitted params to ground truth for overlapping names."""
    rows: List[Dict[str, Any]] = []

    for r in results:
        fitted_params = r["fitted_params"]
        overlap = sorted(set(ground_truth_params) & set(fitted_params))

        if len(overlap) == 0:
            rows.append(
                dict(
                    combo_id=combo_id,
                    repeat_idx=repeat_idx,
                    task_seed=task_seed,
                    model_seed=model_seed,
                    ground_truth_alias=ground_truth_alias,
                    fitted_alias=r["agent_alias"],
                    param_name="__no_overlap__",
                    ground_truth_value=np.nan,
                    fitted_value=np.nan,
                    difference=np.nan,
                    abs_difference=np.nan,
                )
            )
            continue

        for p in overlap:
            gt_val = ground_truth_params[p]
            fit_val = fitted_params[p]

            if isinstance(gt_val, (int, float, np.integer, np.floating)) and isinstance(
                fit_val, (int, float, np.integer, np.floating)
            ):
                diff = float(fit_val) - float(gt_val)
                abs_diff = abs(diff)
            else:
                diff = np.nan
                abs_diff = np.nan

            rows.append(
                dict(
                    combo_id=combo_id,
                    repeat_idx=repeat_idx,
                    task_seed=task_seed,
                    model_seed=model_seed,
                    ground_truth_alias=ground_truth_alias,
                    fitted_alias=r["agent_alias"],
                    param_name=p,
                    ground_truth_value=gt_val,
                    fitted_value=fit_val,
                    difference=diff,
                    abs_difference=abs_diff,
                )
            )

    return rows


def _save_cloudpickle(obj: Any, path: Path) -> None:
    """Save an object with cloudpickle."""
    with path.open("wb") as f:
        cloudpickle.dump(obj, f)


def _save_json(data: Dict[str, Any], path: Path) -> None:
    """Save a JSON file."""
    with path.open("w") as f:
        json.dump(_to_jsonable(data), f, indent=2)


def _save_plot(fig, path: Path) -> None:
    """Save a matplotlib figure and close it."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def _expand_param_grid(param_grid: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Expand a parameter grid into all combinations.

    Scalars are treated as length-1 lists.
    """
    if not param_grid:
        return [{}]

    keys = list(param_grid.keys())
    values_list: List[List[Any]] = []

    for k in keys:
        v = param_grid[k]
        if isinstance(v, np.ndarray):
            values = v.tolist()
        elif isinstance(v, (list, tuple)):
            values = list(v)
        else:
            values = [v]
        values_list.append(values)

    combos = []
    for combo_vals in itertools.product(*values_list):
        combos.append(dict(zip(keys, combo_vals)))
    return combos


def _build_run_dir(
    *,
    output_root: Path,
    gt_alias: str,
    combo_id: int,
    repeat_idx: int,
) -> Path:
    """Create a stable directory for one generator dataset."""
    gt_name = _sanitize_name(gt_alias)
    run_dir = output_root / gt_name / f"combo_{combo_id:05d}" / f"repeat_{repeat_idx:04d}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def _worker_run_single_experiment(
    *,
    combo_id: int,
    gt_param_override: Dict[str, Any],
    repeat_idx: int,
    output_root: str,
    gt_alias: str,
    fitting_aliases: Sequence[str],
    use_uncoupled_task: bool,
    reward_baiting: bool,
    num_trials: int,
    task_seed: int,
    model_seed: int,
    fit_workers: int,
    do_plot_and_save: bool,
    override_existing: bool,   # NEW
) -> Dict[str, Any]:
    """
    One worker unit:
    - instantiate generator
    - set one parameter combination
    - generate one dataset
    - fit all requested aliases
    - save generator/fits
    - return summary rows
    """
    output_root = Path(output_root)

    forager_collection = ForagerCollection()
    df = forager_collection.get_all_foragers()

    if "agent_alias" not in df.columns or "forager" not in df.columns:
        raise ValueError(
            "Expected df from get_all_foragers() to contain columns: "
            "'agent_alias' and 'forager'"
        )

    run_dir = _build_run_dir(
        output_root=output_root,
        gt_alias=gt_alias,
        combo_id=combo_id,
        repeat_idx=repeat_idx,
    )

    # -------------------------------------------------------------------------
    # Generator
    # -------------------------------------------------------------------------
    task = _make_task(
        use_uncoupled=use_uncoupled_task,
        num_trials=num_trials,
        seed=task_seed,
        reward_baiting=reward_baiting,
    )

    forager_gen = _clone_forager_from_df(df, gt_alias, seed=model_seed)
    _set_params_if_requested(forager_gen, gt_param_override)

    forager_gen.perform(task)

    ground_truth_params = _safe_model_dump_params(forager_gen)
    choice_history = np.asarray(forager_gen.get_choice_history())
    reward_history = np.asarray(forager_gen.get_reward_history())

    generator_dir = run_dir / "__generator__"
    generator_dir.mkdir(parents=True, exist_ok=True)

    generator_summary = dict(
        status="ok",
        kind="generator",
        protocol=_get_protocol_name(
        use_uncoupled=use_uncoupled_task,
        reward_baiting=reward_baiting,
    ),
        reward_baiting=reward_baiting,
        num_trials=int(num_trials),
        combo_id=int(combo_id),
        repeat_idx=int(repeat_idx),
        task_seed=int(task_seed),
        model_seed=int(model_seed),
        ground_truth_alias=gt_alias,
        ground_truth_params=ground_truth_params,
        agent_kwargs=getattr(forager_gen, "agent_kwargs", {}),
        params_list_free=getattr(forager_gen, "params_list_free", None),
    )

    _save_json(generator_summary, generator_dir / "summary.json")
    _save_cloudpickle(forager_gen, generator_dir / "fitted_agent.cloudpickle")

    if do_plot_and_save:
        try:
            fig_gen, axes_gen = forager_gen.plot_session(if_plot_latent=True)
            if fig_gen is not None:
                fig_gen.suptitle(f"Generator: {gt_alias}", y=1.02)
                _save_plot(fig_gen, generator_dir / "session_plot.png")
        except Exception:
            pass

    # -------------------------------------------------------------------------
    # Fit all requested aliases
    # -------------------------------------------------------------------------
    fit_summaries: List[Dict[str, Any]] = []
    param_comparison_rows: List[Dict[str, Any]] = []
    results_for_comparison: List[Dict[str, Any]] = []

    for fit_alias in fitting_aliases:

        agent_dir = run_dir / _sanitize_name(fit_alias)
        agent_dir.mkdir(parents=True, exist_ok=True)

        summary_path = agent_dir / "summary.json"

        # ------------------------------------------------------
        # Skip existing fit if override is False
        # ------------------------------------------------------
        if summary_path.exists() and not override_existing:

            with open(summary_path, "r") as f:
                summary = json.load(f)

            fit_summaries.append(
                dict(
                    combo_id=int(combo_id),
                    repeat_idx=int(repeat_idx),
                    task_seed=int(task_seed),
                    model_seed=int(model_seed),
                    protocol=_get_protocol_name(
                        use_uncoupled=use_uncoupled_task,
                        reward_baiting=reward_baiting,
                    ),
                    reward_baiting=reward_baiting,
                    num_trials=int(num_trials),
                    ground_truth_alias=gt_alias,
                    fitted_alias=fit_alias,
                    LPT=summary.get("LPT", np.nan),
                    prediction_accuracy=summary.get("prediction_accuracy", np.nan),
                    n_fit_params=len(summary.get("fit_names", [])),
                    fit_names=";".join(summary.get("fit_names", [])),
                    output_dir=str(agent_dir),
                )
            )

            continue

        # ------------------------------------------------------
        # Otherwise perform fitting
        # ------------------------------------------------------

        fit_agent = _clone_forager_from_df(df, fit_alias, seed=model_seed)

        out = _fit_agent_on_data(
            agent=fit_agent,
            agent_alias=fit_alias,
            choice_history=choice_history,
            reward_history=reward_history,
            seed=model_seed,
            workers=fit_workers,
        )

        results_for_comparison.append(out)

        fr = fit_agent.fitting_result
        agent_dir = run_dir / _sanitize_name(fit_alias)
        agent_dir.mkdir(parents=True, exist_ok=True)

        summary = dict(
            status="ok",
            kind="fit",
            protocol=_get_protocol_name(
                        use_uncoupled=use_uncoupled_task,
                        reward_baiting=reward_baiting,
                    ),
            reward_baiting=reward_baiting,
            num_trials=int(num_trials),
            combo_id=int(combo_id),
            repeat_idx=int(repeat_idx),
            task_seed=int(task_seed),
            model_seed=int(model_seed),
            ground_truth_alias=gt_alias,
            ground_truth_params=ground_truth_params,
            fitted_alias=fit_alias,
            agent_kwargs=getattr(fit_agent, "agent_kwargs", {}),
            pickle_saved=True,
            pickle_backend="cloudpickle",
            pickle_file="fitted_agent.cloudpickle",
            n_trials=int(len(choice_history)),
            LPT=float(out["LPT"]),
            prediction_accuracy=float(out["prediction_accuracy"]),
            fit_names=list(out["fit_names"]),
            x=[float(v) for v in np.asarray(fr.x)],
        )

        if hasattr(fr, "AIC"):
            try:
                summary["AIC"] = float(fr.AIC)
            except Exception:
                pass

        if hasattr(fr, "BIC"):
            try:
                summary["BIC"] = float(fr.BIC)
            except Exception:
                pass

        if hasattr(fr, "fit_settings") and isinstance(fr.fit_settings, dict):
            if "fit_bounds" in fr.fit_settings:
                summary["fit_bounds"] = _to_jsonable(fr.fit_settings["fit_bounds"])

        _save_json(summary, agent_dir / "summary.json")
        _save_cloudpickle(fit_agent, agent_dir / "fitted_agent.cloudpickle")

        if do_plot_and_save:
            try:
                fig_fit, axes_fit = fit_agent.plot_fitted_session()
                if fig_fit is not None:
                    fig_fit.suptitle(
                        f"Fitted: {fit_alias} | "
                        f"LPT={out['LPT']:.4f} | "
                        f"Acc={out['prediction_accuracy']:.4f}",
                        y=1.02,
                    )
                    _save_plot(fig_fit, agent_dir / "fitted_session_plot.png")
            except Exception:
                pass

            try:
                fig_fit2, axes_fit2 = fit_agent.plot_session(if_plot_latent=True)
                if fig_fit2 is not None:
                    fig_fit2.suptitle(
                        f"Observed-history replay: {fit_alias}",
                        y=1.02,
                    )
                    _save_plot(fig_fit2, agent_dir / "observed_history_plot.png")
            except Exception:
                pass

        fit_summaries.append(
            dict(
                combo_id=int(combo_id),
                repeat_idx=int(repeat_idx),
                task_seed=int(task_seed),
                model_seed=int(model_seed),
                protocol=_get_protocol_name(
        use_uncoupled=use_uncoupled_task,
        reward_baiting=reward_baiting,
    ),
                reward_baiting=reward_baiting,
                num_trials=int(num_trials),
                ground_truth_alias=gt_alias,
                fitted_alias=fit_alias,
                LPT=float(out["LPT"]),
                prediction_accuracy=float(out["prediction_accuracy"]),
                n_fit_params=len(out["fit_names"]),
                fit_names=";".join(out["fit_names"]),
                output_dir=str(agent_dir),
            )
        )

    param_comparison_rows = _make_param_comparison_rows(
        ground_truth_alias=gt_alias,
        ground_truth_params=ground_truth_params,
        results=results_for_comparison,
        combo_id=combo_id,
        repeat_idx=repeat_idx,
        task_seed=task_seed,
        model_seed=model_seed,
    )

    return dict(
        status="ok",
        combo_id=int(combo_id),
        repeat_idx=int(repeat_idx),
        task_seed=int(task_seed),
        model_seed=int(model_seed),
        run_dir=str(run_dir),
        fit_summaries=fit_summaries,
        param_comparison_rows=param_comparison_rows,
    )

def _get_protocol_name(*, use_uncoupled: bool, reward_baiting: bool) -> str:
    """Return protocol name from task structure and baiting setting."""
    task_name = "Uncoupled" if use_uncoupled else "Coupled"
    baiting_name = "Baiting" if reward_baiting else "NoBaiting"
    return f"{task_name} {baiting_name}"

    
def run_model_recovery_grid_parallel(
    *,
    output_root: Path,
    ground_truth_forager_alias: str,
    fitting_forager_aliases: Sequence[str],
    ground_truth_param_grid: Optional[Dict[str, Any]],
    n_repeats: int,
    random_seed_base: int,
    task_seed_base: int,
    num_trials: int,
    use_uncoupled_task: bool,
    reward_baiting: bool,
    n_jobs: int,
    fit_workers: int,
    do_plot_and_save: bool,
    override_existing: bool = False,   # NEW
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run model recovery for all parameter combinations x repeats in parallel.

    Returns
    -------
    summary_df
        One row per fitted agent per dataset.
    param_comparison_df
        One row per overlapping parameter comparison.
    """
    output_root.mkdir(parents=True, exist_ok=True)

    param_combos = _expand_param_grid(ground_truth_param_grid)
    jobs: List[Dict[str, Any]] = []

    for combo_id, combo_params in enumerate(param_combos):
        for repeat_idx in range(n_repeats):
            task_seed = task_seed_base + combo_id * 100000 + repeat_idx
            model_seed = random_seed_base + combo_id * 100000 + repeat_idx

            jobs.append(
                dict(
                    combo_id=combo_id,
                    gt_param_override=combo_params,
                    repeat_idx=repeat_idx,
                    output_root=str(output_root),
                    gt_alias=ground_truth_forager_alias,
                    fitting_aliases=list(fitting_forager_aliases),
                    use_uncoupled_task=use_uncoupled_task,
                    reward_baiting=reward_baiting,
                    num_trials=num_trials,
                    task_seed=task_seed,
                    model_seed=model_seed,
                    fit_workers=fit_workers,
                    do_plot_and_save=do_plot_and_save,
                    override_existing=override_existing,  # NEW
                )
            )

    all_fit_summaries: List[Dict[str, Any]] = []
    all_param_comparisons: List[Dict[str, Any]] = []
    failures: List[Dict[str, Any]] = []

    total = len(jobs)
    print(f"0/{total}")

    with ProcessPoolExecutor(max_workers=n_jobs) as ex:
        future_to_job = {
            ex.submit(_worker_run_single_experiment, **job): job
            for job in jobs
        }

        finished = 0

        for fut in as_completed(future_to_job):
            job = future_to_job[fut]
            finished += 1

            try:
                res = fut.result()
                all_fit_summaries.extend(res["fit_summaries"])
                all_param_comparisons.extend(res["param_comparison_rows"])
            except Exception as e:
                failures.append(
                    dict(
                        combo_id=job["combo_id"],
                        repeat_idx=job["repeat_idx"],
                        task_seed=job["task_seed"],
                        model_seed=job["model_seed"],
                        error=repr(e),
                    )
                )

            print(f"{finished}/{total}")

    summary_df = pd.DataFrame(all_fit_summaries)
    if len(summary_df) > 0:
        summary_df = summary_df.sort_values(
            ["combo_id", "repeat_idx", "LPT", "prediction_accuracy"],
            ascending=[True, True, False, False],
        )

    param_comparison_df = pd.DataFrame(all_param_comparisons)
    if len(param_comparison_df) > 0:
        param_comparison_df = param_comparison_df.sort_values(
            ["combo_id", "repeat_idx", "fitted_alias", "param_name"]
        )

    summary_df.to_csv(output_root / "all_fit_summaries.csv", index=False)
    param_comparison_df.to_csv(output_root / "all_param_comparisons.csv", index=False)

    failure_df = pd.DataFrame(failures)
    failure_df.to_csv(output_root / "failures.csv", index=False)

    run_config = dict(
        output_root=str(output_root),
        ground_truth_forager_alias=ground_truth_forager_alias,
        fitting_forager_aliases=list(fitting_forager_aliases),
        ground_truth_param_grid=_to_jsonable(ground_truth_param_grid),
        n_repeats=int(n_repeats),
        random_seed_base=int(random_seed_base),
        task_seed_base=int(task_seed_base),
        num_trials=int(num_trials),
        use_uncoupled_task=bool(use_uncoupled_task),
        reward_baiting=bool(reward_baiting),
        n_jobs=int(n_jobs),
        fit_workers=int(fit_workers),
        do_plot_and_save=bool(do_plot_and_save),
        n_param_combinations=int(len(param_combos)),
        n_total_experiment_units=int(len(jobs)),
        n_failures=int(len(failures)),
        override_existing=bool(override_existing),
    )
    _save_json(run_config, output_root / "run_config.json")

    return summary_df, param_comparison_df




In [2]:
# =============================================================================
# User config
# =============================================================================

OUTPUT_ROOT = Path("/root/capsule/scratch/model_recovery_parallel")

RANDOM_SEED_BASE = 42
TASK_SEED_BASE = 53

NUM_TRIALS = 1000
USE_UNCOUPLED_TASK = True
REWARD_BAITING = False

GROUND_TRUTH_FORAGER_ALIAS = "QLearning_L1F1_CK1_softmax"

FITTING_FORAGER_ALIASES = [
    "QLearning_L1F1_CK1_softmax",
    "QLearning_L2F1_CK1_softmax",
    "ForagingCompareThreshold_L2_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF",
    "ForagingCompareThreshold_L1_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF",
]

# Each value can be:
# - a scalar
# - a list / tuple / np.ndarray
#
# All combinations will be expanded.
GROUND_TRUTH_PARAM_GRID: Dict[str, Any] = {
    "learn_rate": np.linspace(0.1, 0.86, 4),
    "forget_rate_unchosen": np.linspace(0.0, 1.0, 4),
    "choice_kernel_relative_weight": np.linspace(0.0, 1.0, 4),
    "softmax_inverse_temperature": np.linspace(0.1, 20.0, 4),
    "biasL": np.linspace(-2.0, 2.0, 4),
}

N_REPEATS = 20

# Parallelization across experiment units: (param combination) x (repeat)
N_JOBS = 120

# Inner DE workers for each fit.
# When running many jobs in parallel, keep this at 1 to avoid nested oversubscription.
FIT_WORKERS = 1

DO_PLOT_AND_SAVE = False



# =============================================================================
# Run
# =============================================================================

if __name__ == "__main__":
    summary_df, param_comparison_df = run_model_recovery_grid_parallel(
        output_root=OUTPUT_ROOT,
        ground_truth_forager_alias=GROUND_TRUTH_FORAGER_ALIAS,
        fitting_forager_aliases=FITTING_FORAGER_ALIASES,
        ground_truth_param_grid=GROUND_TRUTH_PARAM_GRID,
        n_repeats=N_REPEATS,
        random_seed_base=RANDOM_SEED_BASE,
        task_seed_base=TASK_SEED_BASE,
        num_trials=NUM_TRIALS,
        use_uncoupled_task=USE_UNCOUPLED_TASK,
        reward_baiting=REWARD_BAITING,
        n_jobs=N_JOBS,
        fit_workers=FIT_WORKERS,
        do_plot_and_save=DO_PLOT_AND_SAVE,
    )

    print("\n================ AGGREGATED FIT SUMMARY ================\n")
    if len(summary_df) > 0:
        print(summary_df.head(20).to_string(index=False))
    else:
        print("No successful fits.")

    print("\n================ AGGREGATED PARAM COMPARISON ================\n")
    if len(param_comparison_df) > 0:
        print(param_comparison_df.head(20).to_string(index=False))
    else:
        print("No parameter comparison rows.")

0/20480


2026-03-07 21:11:42,106 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,173 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,174 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,187 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,253 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,262 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,266 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:11:42,272 - aind_dynamic_foraging_models.generative_model.base - INFO

1/20480


2026-03-07 21:19:43,849 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:19:43,856 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:19:45,810 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


2/20480


...

2026-03-07 21:24:02,503 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:11,421 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


83/20480


2026-03-07 21:24:13,207 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:13,224 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:13,611 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:13,701 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:19,132 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:20,125 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:22,104 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:24:24,433 - aind_dynamic_foraging_models.generative_model.base - INFO

84/20480


2026-03-07 21:25:12,076 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:25:13,346 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:25:14,489 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-03-07 21:25:17,475 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
